## Clone repo

In [ ]:
!git clone https://github.com/dwctien/diffusion-image-inpainting.git
%cd diffusion-image-inpainting

## Install dependencies

In [ ]:
!pip install -r requirements.txt

## Check GPU

In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Test imports

In [ ]:
from src.utils import load_config, ensure_output_dirs
from src.preprocessing import load_image, load_mask, prepare_image_and_mask, create_masked_image
from src.mask_generator import generate_mask
from src.model_loader import load_pipeline_from_config
from src.pipeline import run_inpainting_from_paths
from src.metrics import compute_all_metrics

print("All imports successful.")

## Load config + create output dirs

In [ ]:
config = load_config("configs/default.yaml")
ensure_output_dirs(config)

config

## Test preprocessing + mask generation

In [ ]:
from pathlib import Path
from src.preprocessing import load_image, prepare_image_and_mask, create_masked_image
from src.mask_generator import generate_mask
from src.utils import save_image

image_path = Path("/kaggle/input/datasets/dwctien/places365-test/Places365_test_00000001.jpg")

image = load_image(image_path)
mask = generate_mask(size=image.size, mask_type="rectangle", mask_ratio=0.25)

image, mask = prepare_image_and_mask(image, mask, image_size=512)
masked = create_masked_image(image, mask)

save_image(image, "outputs/visualizations/test_original.png")
save_image(mask, "outputs/visualizations/test_mask.png")
save_image(masked, "outputs/visualizations/test_masked.png")

print(image.size, mask.size, masked.size)

## Test load model

In [ ]:
from src.model_loader import load_pipeline_from_config

pipe = load_pipeline_from_config(config)
print("Pipeline loaded.")
print(pipe.device)

## Test single-image inference

In [ ]:
!python scripts/run_inference.py \
  --config configs/default.yaml \
  --image ... \
  --mask outputs/visualizations/test_mask.png \
  --prompt "a realistic photo" \
  --output outputs/images/test_result.png \
  --save_masked_preview

## Test small evaluation

In [ ]:
!python scripts/run_evaluation.py \
  --config configs/default.yaml \
  --input_dir /kaggle/input/datasets/dwctien/places365-test \
  --output_dir outputs \
  --num_samples 5 \
  --mask_type rectangle \
  --prompt "a realistic photo" \
  --save_visualizations

In [ ]:
import pandas as pd

df = pd.read_csv("outputs/metrics/metrics.csv")
df.head()